# Assignment 1: Some Numerical Model data 

# 1. Lorenz63 System$^{[1]}$

The first model is the Lorenz system which was derived by Edward Lorenz in 1963 as a simplified system of atmospheric convection. It is a system of 3 coupled nolinear ordinary differential equations (ODEs) that described fluid dynamics in the atmosphere and chaos theory, meaning its output can be highly sensitive to small changes in its starting conditions. The model has 3 state variables and 3 parameters$^{[2]}$

| Symbol | Name | Physical interpretation |
| :--- | :--- | :--- |
| $x(t)$ | Intensity of convection | Speed and direction of teh convection |
| $y(t)$ | Horizontal temperature difference| Teamperature difference between different location |
| $z(t)$ | Vertical temperature gradient | Vertical temperature variation from linear |
| $\sigma$ | Prandtl number | Ratio of momentum diffusivity to thermal diffusivity |
| $\rho$ | Rayleigh number | Control convection strength |
| $\beta$ | Geometric factor | Relate to the physical dimensions of teh fluid layer |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.integrate import odeint
from scipy.fftpack import diff as psdiff
from mpl_toolkits.mplot3d import Axes3D

def lorenz63(t, state, sigma, rho, beta):
    x, y, z = state
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z
    return [dx, dy, dz]

## 1.1 Observed behaviours

### 1.1.1 The Lorenz attractor (Butterfly shape)

The figure shows the spatial trajectory of the Lorenz63 system with classic parameters ($\sigma$=10, $\rho$=28, $\beta$=8/3). The trajectory shows the  butterfly shape and the system's state does not repeat over time, but it is constrained to a finite spatial region. To visually illustrate the movement over time, a color gradient (blue represents the early stage, red represents the later stage) is used to visually demonstrate the continuous alternation of the pipe between the two wings. The two wings rotate around different centers and connect through a narrow central region.

In [ ]:
state0 = [1.0, 1.0, 1.0]  # Initial state
params = (10.0, 28.0, 8.0/3.0)  # sigma, rho, beta

t = np.linspace(0, 50, 5000)
sol = solve_ivp(lorenz63, [t[0], t[-1]], [0.1, 0.1, 0.1], t_eval=t, args=params)

x, y, z = sol.y

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(x, y, z, c=t, cmap='coolwarm', s=1)
plt.colorbar(sc, ax=ax, label='Time')
ax.set_xlabel(r"$x(t)$")
ax.set_ylabel(r"$y(t)$")
ax.set_zlabel(r"$z(t)$")
plt.title(f"Lorenze attractor \nsigma={params[0]}, rho={params[1]}, beta={params[2]}", fontsize=15)

### 1.1.2 Time series

The figure shows the three state variables as time seties. All variables exhibit oscillations with no obvious periood. The system switches irregularly between a phase where $x$ and $y$ oscillate with large positive (right wing) and with large negative (left wing). The $z$ variable always stay in positive. 

In the early phase, variables rapidly grow from initial conditions before stabilizing into chaotic oscillations. After $t>15$ , variables show irregular oscillation frequencies and amplitudes. The system switches between wings unpredictably.

In [ ]:
fig = plt.figure(figsize=(10, 4))

ax = plt.subplot(3, 1, 1)
ax.plot(t, x, "C0")
ax.grid()
ax.set_ylabel(r"$x(t)$")
ax.set_xticklabels([])

ax = plt.subplot(3, 1, 2)
ax.plot(t, y, "C1")
ax.grid()
ax.set_ylabel(r"$y(t)$")
ax.set_xticklabels([])

ax = plt.subplot(3, 1, 3)
ax.plot(t, z, "C2")
ax.grid()
ax.set_ylabel(r"$z(t)$")
ax.set_xlabel(r"$t$")

### 1.1.3 Sensitive dependence on initial conditions

The figure shows the popular concept od the butterfly effect that like the flap of a butterfly's wings, could ultimately alter large-scale weather patterns. Here, I added an initial error of $10^{-8}$ to $x(t)$ in the initial conditions. 

The results show that the time series of $x(t)$ can yield relatively accurate results in the range of $t < 40$, but the results become completely different beyond this range.

In [ ]:
eps = 1e-8
sol1 = solve_ivp(lorenz63, [t[0], t[-1]], [0.1, 0.1, 0.1], t_eval=t, args=params)
sol2 = solve_ivp(lorenz63, [t[0], t[-1]], [0.1 + eps, 0.1, 0.1], t_eval=t, args=params)

x1, y1, z1 = sol1.y
x2, y2, z2 = sol2.y

fig, ax = plt.subplots(1, 1, figsize=(10, 3), sharex=True)
ax.plot(t, x1, label='Trajectory 1', lw=1)
ax.plot(t, x2, label='Trajectory 2', lw=1, linestyle='--')
ax.set_ylabel('$x(t)$')
ax.legend()
ax.set_title("Butterfly effect", fontsize=20)

### 1.1.4 Parameter variation ($\rho$)

To study the effect of rho on the Lorenz system, I plotted graphs using four different $\rho$ values. The graph shows that when $\rho$=10, the butterfly pattern did not appear, and the trajectory was distributed in the range $x > 0$. When $\rho=20$, the trajectory was mainly distributed in the range $x < 0$. However, when $\rho = 28$ and $\rho = 99$, the aforementioned butterfly pattern appeared.

In [ ]:
rho_vals = [10, 20, 28, 99]

fig = plt.figure(figsize=(12, 9))
for i, rho in enumerate(rho_vals):
    sol = solve_ivp(lorenz63, [t[0], t[-1]], [0.1, 0.1, 0.1], t_eval=t, args=(10.0, rho, 8.0/3.0))
    xi, yi, zi = sol.y
    ax = fig.add_subplot(2, 2, i+1, projection='3d')
    ax.scatter(xi, yi, zi, c=t, cmap="coolwarm", s=1)
    ax.set_title(f'rho = {rho}')
    ax.set_xlabel('$x(t)$')
    ax.set_ylabel('$y(t)$')
    ax.set_zlabel('$z(t)$')
plt.suptitle("rho variation", fontsize=20)
plt.tight_layout()

## 1.2 Sklearn Testing

### 1.2.1 Wing classification

The system alternates between two wings of the attractor that the left wings means $x < 0$ with counterclockwise for the flow and the right wings means $x > 0$ with clockwise for the flow. **Here, I attempt to classify the system's location using different methods.** Below, I use the observations of two variables, $y$ and $z$, at two time points to predict the current state. First, I generate enough sample data for machine learning. The values ​​of the sample data are then evaluated: data where $x > 0$ is converted to 1 (representing the right wing), and data where $x < 0$ is converted to 0 (representing the left wing).

In [ ]:
t_long = np.linspace(0, 200, 20000)
sol_long = solve_ivp(lorenz63, [t_long[0], t_long[-1]], [0.1, 0.1, 0.1], t_eval=t_long, args=params)
xl, yl, zl = sol_long.y
N = xl.shape[0]
wing = (xl > 0).astype(int)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
for label, color, name in [(0, 'blue', 'Left wing (x<0)'), (1, 'red', 'Right wing(x>0)')]:
    mask = wing == label
    ax.scatter(xl[mask], yl[mask], zl[mask], c=color, s=0.1, alpha=0.4, label=name)
ax.legend(markerscale=10)
ax.set_xlabel('$x(t)$')
ax.set_ylabel('$y(t)$')
ax.set_zlabel('$z(t)$')

First, I finish the classification task. Since the sign of $x(t)$ represents the wing where the model is located, I use $y(t)$ and $z(t)$ to represent the model state at $t$, and $y(t-\tau)$ and $z(t-\tau)$ to represent the model state at $t-\tau$. The result labels are the values marked above. The model will predict which wing the model will be located in at $t+\tau$ based on the states at $t$ and $t-\tau$.

I used the confusion matrix as the parameter to to judge the model's accuracy. The figure shows the confusion matrix results using the random forest classifier and the Gradient Boosting classifier, respectively. As you can see, both classifiers effectively identifying the left and right wings of the Lorenz attractor.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, r2_score, mean_squared_error

tau = 5
delta = 5
start = tau
end = N - delta

x_clss = np.column_stack([yl[start:end], zl[start:end], yl[start-tau:end-tau], zl[start-tau:end-tau]])
y_clss = wing[start+delta : end+delta]

scaler  = StandardScaler()
x_clss_2 = scaler.fit_transform(x_clss)

x_train, x_test, y_train, y_test = train_test_split(x_clss_2, y_clss, test_size=0.2, stratify=y_clss, random_state=42)

rf_clss = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
gb_clss = GradientBoostingClassifier(n_estimators=200, max_depth=15, learning_rate=0.1, random_state=42)
rf_clss.fit(x_train, y_train)
gb_clss.fit(x_train, y_train)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, model, title in zip(axes, [rf_clss, gb_clss], ['Random Forest', 'Gradient Boosting']):
    metrix = confusion_matrix(y_test, model.predict(x_test))
    display = ConfusionMatrixDisplay(metrix, display_labels=['Left', 'Right'])
    display.plot(ax=ax, colorbar=False)
    accuracy = model.score(x_test, y_test)
    ax.set_title(f'{title}\nAccuracy = {accuracy:.4f}')
plt.tight_layout()

### 1.2.2 Regression

**I tried to predict the values of $x$of the model at $t+\delta t$ using the complete model state at $t$.** Here I use $R^2$ and $RMSE$ as the parameters to determine the model's accuracy. 

First, I set the $\delta t = 50$, it shows both model can obtain great predict results. When I set the $\delta t = 500$, the model's accuracy became very poor, which corresponds to the butterfly effect mentioned earlier. **This demonstrates that short-term predictions are feasible in the Lorenz63 system, but the prediction performance degrades rapidly as the $\delta t$ increases.**

In [ ]:
delta_reg = 50
end_reg   = N - delta_reg

x_reg = np.column_stack([xl[:end_reg], yl[:end_reg], zl[:end_reg]])
y_reg = xl[delta_reg:]

x_train_r, x_test_r, y_train_r, y_test_r = train_test_split(x_reg, y_reg, test_size=0.2, random_state=42)

rf_reg = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
gb_reg = GradientBoostingRegressor(n_estimators=200, max_depth=15, learning_rate=0.1, random_state=42)

rf_reg.fit(x_train_r, y_train_r)
gb_reg.fit(x_train_r, y_train_r)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, model, title in zip(axes, [rf_reg, gb_reg], ['Random Forest', 'Gradient Boosting']):
    pred = model.predict(x_test_r)
    r2  = r2_score(y_test_r, pred)
    rmse = np.sqrt(mean_squared_error(y_test_r, pred))
    ax.scatter(y_test_r, pred, s=1, alpha=0.3, color='blue')
    lims = [y_test_r.min(), y_test_r.max()]
    ax.plot(lims, lims, 'r--', lw=1.5)
    ax.set_xlabel('True x(t+Δt)')
    ax.set_ylabel('Predicted x(t+Δt)')
    ax.set_title(f'{title} ($\delta t$ = 50) \nR²={r2:.4f},  RMSE={rmse:.4f}')
plt.tight_layout()

In [ ]:
delta_reg2 = 500
end_reg2   = N - delta_reg2

x_reg2 = np.column_stack([xl[:end_reg2], yl[:end_reg2], zl[:end_reg2]])
y_reg2 = xl[delta_reg2:]

x_train_r2, x_test_r2, y_train_r2, y_test_r2 = train_test_split(x_reg2, y_reg2, test_size=0.2, random_state=42)

rf_reg2 = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
gb_reg2 = GradientBoostingRegressor(n_estimators=200, max_depth=15, learning_rate=0.1, random_state=42)

rf_reg2.fit(x_train_r2, y_train_r2)
gb_reg2.fit(x_train_r2, y_train_r2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, model, title in zip(axes, [rf_reg2, gb_reg2], ['Random Forest', 'Gradient Boosting']):
    pred2 = model.predict(x_test_r2)
    r2  = r2_score(y_test_r2, pred2)
    rmse = np.sqrt(mean_squared_error(y_test_r2, pred2))
    ax.scatter(y_test_r2, pred2, s=1, alpha=0.3, color='blue')
    lims = [y_test_r2.min(), y_test_r2.max()]
    ax.plot(lims, lims, 'r--', lw=1.5)
    ax.set_xlabel('True x(t+Δt)')
    ax.set_ylabel('Predicted x(t+Δt)')
    ax.set_title(f'{title} ($\delta t$ = 500) \nR²={r2:.4f},  RMSE={rmse:.4f}')
plt.tight_layout()

### 1.2.3 Impact of different hyperparameters to the model

To gain a more detailed understanding of the impact of hyperparameters, I manually adjusted the key hyperparameters in the model, including **n_estimators**, **max_depth** and **learining_rate**. Instead of using default values, I performed a manual grid search by changing only one parameter at a time while keeping the others constant.$^{[3]}$

In [ ]:
estimators_vals = [10, 20, 30, 50, 70, 100, 150, 200]
rf_acc_estimators = []
for e in estimators_vals:
    model = RandomForestRegressor(n_estimators=e, max_depth=15, random_state=42, n_jobs=-1)
    model.fit(x_train_r, y_train_r)
    acc = r2_score(y_test_r, model.predict(x_test_r))
    rf_acc_estimators.append(acc) 

plt.figure(figsize=(7, 4))
plt.plot(range(len(estimators_vals)), rf_acc_estimators, 'o-', color='#2E75B6', lw=2)
plt.xticks(range(len(estimators_vals)), estimators_vals)
plt.xlabel('n_estimators')
plt.ylabel('Accuracy')
plt.title('RF (n_estimators)')
plt.grid(alpha=0.4)

#### RF: n_estimators (The number of decision trees)

When n_estimators = 20, accuracy reaches its minimum. Accuracy improves significantly with increasing number of tree, peaking around 99.804% at n_estimators = 100-150. Beyond 150 trees, accuracy slightly decreases to 99.803%, showing mild overfitting tendencies.

In [ ]:
depth_vals = [1, 2, 3, 5, 7, 10, 15, 20]
rf_acc_depth = []
for d in depth_vals:
    model = RandomForestRegressor(n_estimators=200, max_depth=d, random_state=42, n_jobs=-1)
    model.fit(x_train_r, y_train_r)
    acc = r2_score(y_test_r, model.predict(x_test_r))
    rf_acc_depth.append(acc) 

plt.figure(figsize=(7, 4))
plt.plot(range(len(depth_vals)), rf_acc_depth, 'o-', color='#2E75B6', lw=2)
plt.xticks(range(len(depth_vals)), depth_vals)
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('RF (max_depth)')
plt.grid(alpha=0.4)

#### RF: max_depth

Accuracy increases with max_depth increases. At max_depth = 1-2, accuracy is around 50%, indicating severe underfitting where the model cannot capture data patterns. At max_depth ≥ 10, accuracy exceeds 99.9% and stabilizes. However, deeper trees (15, 20) do not significantly improve accuracy but increase computational cost and overfitting risk.

In [ ]:
estimators_vals = [10, 20, 30, 50, 70, 100, 150, 200]
gb_acc_estimators = []
for e in estimators_vals:
    model = GradientBoostingRegressor(n_estimators=e, max_depth=15, learning_rate=0.1, random_state=42)
    model.fit(x_train_r, y_train_r)
    acc = r2_score(y_test_r, model.predict(x_test_r))
    gb_acc_estimators.append(acc) 

plt.figure(figsize=(7, 4))
plt.plot(range(len(estimators_vals)), gb_acc_estimators, 'o-', color='#2E75B6', lw=2)
plt.xticks(range(len(estimators_vals)), estimators_vals)
plt.xlabel('n_estimators')
plt.ylabel('Accuracy')
plt.title('Gradient Boosting (n_estimators)')
plt.grid(alpha=0.4)

#### Gradient Boosting: n_estimators

At n_estimators = 10, accuracy is only around 87%. From n_estimators = 20, accuracy increase sharply to 98%. At n_estimators ≥ 30, accuracy stabilizes above 99%. Gradient Boosting converges faster with only 30 trees needed for best performance versus 100-150 for Random Forest.

In [ ]:
depth_vals = [1, 2, 3, 5, 7, 10, 15, 20]
gb_acc_depth = []
for d in depth_vals:
    model = GradientBoostingRegressor(n_estimators=200, max_depth=d, learning_rate=0.1, random_state=42)
    model.fit(x_train_r, y_train_r)
    acc = r2_score(y_test_r, model.predict(x_test_r))
    gb_acc_depth.append(acc) 

plt.figure(figsize=(7, 4))
plt.plot(range(len(depth_vals)), gb_acc_depth, 'o-', color='#2E75B6', lw=2)
plt.xticks(range(len(depth_vals)), depth_vals)
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Gradient Boosting (max_depth)')
plt.grid(alpha=0.4)

#### Gradient Boosting: max_depth

Gradient Boosting similarly shows accuracy improving with increasing max_depth. At max_depth = 1, accuracy is only about 62%. At max_depth = 2, accuracy jump to 87% and at max_depth = 3, it reaches over 95%. At max_depth ≥ 5, accuracy stabilizes at nearly 100% and remains constant.

In [ ]:
lr_vals = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1]
gb_acc_lr = []
for l in lr_vals:
    model = GradientBoostingRegressor(n_estimators=200, max_depth=15, learning_rate=l, random_state=42)
    model.fit(x_train_r, y_train_r)
    acc = r2_score(y_test_r, model.predict(x_test_r))
    gb_acc_lr.append(acc) 

plt.figure(figsize=(7, 4))
plt.plot(range(len(lr_vals)), gb_acc_lr, 'o-', color='#2E75B6', lw=2)
plt.xticks(range(len(lr_vals)), lr_vals)
plt.xlabel('learning_rate')
plt.ylabel('Accuracy')
plt.title('Gradient Boosting (learning_rate)')
plt.grid(alpha=0.4)

#### Gradient Boosting: learning_rate

At learning_rate = 0.001, accuracy is only 33%, indicating slow learning where tiny step sizes prevent convergence to the great solutions. At learning_rate = 0.005, accuracy rapidly increases to over 85%. At learning_rate ≥ 0.05, accuracy stabilizes at over 99%. This shows that learning rate must exceed a certain threshold for Gradient Boosting to learn effectively. 

# 2. Korteweg–de Vries equation (KdV)$^{[4]}$

The KdV equation is a nonlinear partial differential equation governing the evolution of shallow water wave height $u(x,t)$ in one spatial dimension. It was originally derived in 1895 by Korteweg and de Vries to explain the stable, hump-shaped waves observed by John Scott Russell in canals which is known as solitons.$^{[5]}$
| Symbol | Name | Physical interpretation |
| :--- | :--- | :--- |
| $u(x,t)$ | Wave height | The wave height in position $x$ and time $t$ |
| $x$ | Spatial coordinate | Horizontal position along the domain $[0, L)$ |
| $t$ | Time  | Evolution time of the simulation |
| $c$ | Speed parameter | Determines the amplitude and the width of the soliton |
| $L$ | Domain length | The total spatial extent of the periodic boundary |
| $N$ | Number of grid points | Number of spatial discretisation points |

In [ ]:
def kdv_exact(x, c):
    u = 0.5*c*np.cosh(0.5*np.sqrt(c)*x)**(-2)
    return u

def kdv(u, t, L):
    ux = psdiff(u, period=L)
    uxxx = psdiff(u, period=L, order=3)
    return -6*u*ux - uxxx

L = 50.0
N = 64
dx = L / (N - 1.0)
x = np.linspace(0, (1-1.0/N)*L, N)

T = 200
t = np.linspace(0, T, 501)

u0 = kdv_exact(x-0.33*L, 0.75) + kdv_exact(x-0.65*L, 0.4)
sol = odeint(kdv, u0, t, args=(L,), mxstep=5000)

## 2.1 Observed behaviours

### 2.1.1 Parameter variation ($c$)

The code above illustrates a hypothetical numerical experiment in which the initial condition consists of two solitons with different value of $c$. Firstly, I want to know what different situation correspond to different values of $c$.

In [ ]:
x_single = np.linspace(-10, 10, 300)
c_values = [0.4, 0.75]

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
for ax, c in zip(axes, c_values):
    u = kdv_exact(x_single, c)
    ax.plot(x_single, u, linewidth=2)
    ax.set_title(f'$c = {c}$', fontsize=13)
    ax.set_xlabel('$x$', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.6)

axes[0].set_ylabel('$u(x)$', fontsize=12)
fig.suptitle('KdV Soliton Profile: Effect of Speed Parameter $c$', fontsize=13)
plt.tight_layout()

### 2.1.2 Experiment setting


#### Initial conditions:

The figure illustrates a hypothetical numerical experiment in which the initial condition consists of two solitons with different amplitudes and speeds. Specifically, a faster, taller soliton ($c$ = 0.75) placed behind a slower, shorter one ($c$ = 0.4), which in the diagram manifests as two localised bright peaks at $t$ = 0, positioned at approximately $x$ = 16.5 and $x$ = 32.5 respectively. The spatial domain is set with a periodic right boundary of length $L$ = 50, so that any wave passing through the right boundary re-enters seamlessly from the left, meaning the solitons circulate continuously without energy loss or reflection. 

#### Movement process:

In the Hovmoller plot, the motion of the solitons are represented by diagonal stripes, with the slope representing velocity. Because the faster soliton ($c$ = 0.75, brighter one) travels faster than the slower one ($c$ = 0.40), it continuously closes the gap between them. The first interation occurs at approximately 20~30 s, visible in the Hovmoller diagram as the two stripes collide. After re-separation, both solitons continue circulating, and owing to the periodic boundary the faster soliton laps the slower one repeatedly. 

As shown in the plot, after the interaction occurs, the taller solidton appears in front of the shorter one, and the movement speed of both solidtons gradually returns to the initial state (the slope remains the same).

In [ ]:
# Hovmoller plot (time going up; have a think about what this is showing)
fig = plt.figure(figsize=(6, 5))
ax = plt.axes()
cs = ax.contourf(x, t, sol, levels=101)
cax = plt.colorbar(cs)
cax.set_label('Wave height $u(x,t)$', fontsize=12)
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$t$')

### 2.1.3 Different time snapshots

Snapshots at t = 0, 20, 40 show the two pulses before, during, and after interaction. The curve t=0 represents the initial conditions, consistent with what was mentioned above. The position at x = 16.5 represents the taller soliton (c=0.75) and the position at x=32.5 represents the shorter soliton (c=0.4). 

The curve t = 20 represents the situation where the two solitons are about to intersect. It can be seen that the two solitons are rapidly approaching each other. The wave height of the soliton in the taller one decreases slightly, while the wave height of the soliton in the shorter increases slightly.

The curve t=40 represents the situation after the two solitons intersect. It can see that the two solitons have separated, the taller soliton have crossed the right boundary and appeared on the left, while the shorter solitons remain on the right. The wave heights of both solitons remain almost unchanged. It's worth noting that by comparing the positions of the shorter solitons, after the same 20 seconds, the distance traveled in the first 20 seconds is greater than that in the 20-40 seconds. This indicates that at the time of the intersect, the taller solitons moved forward, while the shorter solitons' movement was hindered.

In [ ]:
# snapshots in time (have a think about what this is showing)
Nt = len(t)
inds = [0, Nt // 10, Nt // 5]

fig = plt.figure(figsize=(6, 5))
ax = plt.axes()
for i in range(len(inds)):
    ax.plot(x, sol[inds[i], :], label=f"$t = {{{t[inds[i]]:.2f}}}$", alpha=0.7)
ax.legend()
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$u(x, t=t_0)$')
ax.grid()

### 2.1.4 Maximum wave height in the model

The figure shows the maximum wave height during the experiment. It can be seen that when the two solitons do not intersect, the maximum wave height is equal to that of the taller soliton and remains relatively stable. When the two solitons intersect, the maximum wave height is less than that of the taller soliton.

In [ ]:
soliton1 = 0.5 * 0.75
epsilon = soliton1 - 0.01
max_u = sol.max(axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, max_u, color='royalblue', linewidth=2, label='max $u$')
ax.axhline(soliton1, color='green', linestyle=':', linewidth=1.5, label=f'Amplitude = {soliton1:.3f} (taller soliton)')
ax.axhline(epsilon, color='red', linestyle=':', linewidth=1.5)

ax.set_xlabel('$t$', fontsize=12)
ax.set_ylabel('max $u$', fontsize=12)
ax.legend()
ax.grid(True, linestyle='--')
plt.tight_layout()

## 2.2 Sklearn Testing

Since wave height does not change significantly when no interaction occurs, it is difficult to predict future conditions based on wave height at a single moment. Therefore, **I trying to use wave heights from multiple past moments together to predict the wave height at the next moment.**

### 2.2.1 Different models comparision$^{[3]}$

Firstly, I set the number of past time steps to 5 as an example for model building. When setting the model input, I obtain the sol values of 5 consecutive time steps and flatten them into one dimension. Since the first 5 time steps are only used as input, I get a total of 495 rows of data, meaning the dimension of x is (495, 320). Similarly, the dimension of y is (496, 64). 

Then, I set 70% of the data for model training and 30% for validation. Considering that past data should be used to predict the future, future data should not be included in the training. Therefore, the parameter "shuffle" in train_test_split is set to False to maintain the original order of the data.

I tested three regression models, **Ridge Regression**, **RandomForest Regression** and **GradientBoosting Regression**. The output should be a result of (1, 64). Ridge Regression and RandomForest Regression both have this capability, while GradientBoosting Regression requires the use of MultiOutputRegressor.

In [ ]:
window = 5

x_list = []
y_list = []

# i = 0, 1, 2, ..., 495
# i=0: xi = [t0,t1,t2,t3,t4], yi = t5
# i=1: xi = [t1,t2,t3,t4,t5], yi = t6
for i in range(len(t) - window):
    xi = sol[i:i+window, :].flatten()
    yi = sol[i+window, :]
    x_list.append(xi)
    y_list.append(yi)

x_list = np.array(x_list)
y_list = np.array(y_list) 
x_train_kdv, x_test_kdv, y_train_kdv, y_test_kdv = train_test_split(x_list, y_list, test_size=0.3, shuffle=False)
split = len(x_train_kdv)
t_test = t[window + split:]

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor

ridge = Ridge(alpha=1.0)
ridge.fit(x_train_kdv, y_train_kdv)
y_pred_ridge = ridge.predict(x_test_kdv)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(x_train_kdv, y_train_kdv)
y_pred_rf = rf.predict(x_test_kdv)

gb_base = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model = MultiOutputRegressor(gb_base, n_jobs=-1)
gb_model.fit(x_train_kdv, y_train_kdv)
y_pred_gb = gb_model.predict(x_test_kdv)

In [ ]:
r2_ridge = r2_score(y_test_kdv, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test_kdv, y_pred_ridge))

r2_rf = r2_score(y_test_kdv, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_kdv, y_pred_rf))

r2_gb = r2_score(y_test_kdv, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test_kdv, y_pred_gb))

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

figure = [(axes[0,0], y_test_kdv, 'A. True $u(x,t)$', None),
          (axes[0,1], y_pred_ridge, f'B. Ridge \n $R^2={r2_ridge:.3f}$ $RMSE={rmse_ridge:.3f}$', {r2_ridge, rmse_ridge}),
          (axes[1,0], y_pred_rf, f'C. Random Forest \n $R^2={r2_rf:.3f}$ $RMSE={rmse_rf:.3f}$', {r2_rf, rmse_rf}), 
          (axes[1,1], y_pred_gb, f'D. Gradient Boosting \n $R^2={r2_gb:.3f}$ $RMSE={rmse_gb:.3f}$', {r2_gb, rmse_gb})]

for ax, data, title, r2 in figure:
    cs = ax.contourf(x, t_test, data, levels=101, vmin=0, vmax=0.37)
    plt.colorbar(cs, ax=ax, label='$u(x,t)$')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$t$')
    ax.set_title(title, fontsize=12, fontweight='bold')
plt.tight_layout()

### 2.2.2 Impact of hyperparameters ($alpha$) to the model

Similar to the above, here I tested the parameter $alpha$ in Ridge regression, keeping other parameters unchanged and performed a manual grid search. The $alpha$ represents the regularization strength of the model.

The results demonstrate that the model maintains high accuracy (approximately 1.0) across a wide range of small $alpha$ values (0.0001 to 1), indicating that minimal regularization is needed for this task. However, as alpha increases beyond 1, the model's performance deteriorates significantly, with accuracy dropping from 0.95 at $alpha$ = 10 to approximately 0.13 at $alpha$ = 1000.

In [ ]:
alpha_vals = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000]
ridge_acc_alpha = []
for a in alpha_vals:
    model = Ridge(alpha=a)
    model.fit(x_train_kdv, y_train_kdv)
    acc = r2_score(y_test_kdv, model.predict(x_test_kdv))
    ridge_acc_alpha.append(acc) 

plt.figure(figsize=(7, 4))
plt.plot(range(len(alpha_vals)), ridge_acc_alpha, 'o-', color='#2E75B6', lw=2)
plt.xticks(range(len(alpha_vals)), alpha_vals)
plt.xlabel('alpha')
plt.ylabel('Accuracy')
plt.title('Ridge (alpha)')
plt.grid(alpha=0.4)

### 2.2.3 Window size effect on Regression Performance

Next, I want to see how the accuracy of the model changes when I use different windows size. Here using the ridge model for this test. The analysis tests window sizes ranging from 1 to 20 time steps, comparing predicted solutions against the ground truth. 

Figure A shows the true $u(x,t)$, while figure B-H demonstrate predictions using increasingly larger window size. The results shows that window sizes of 3 and 5 time steps achieve best performance, with R² = 0.991 and 0.989 respectively, RMSE = 0.009 for both. Smaller windows (1-2 steps) and larger windows (10-20 steps) show slightly degraded performance. 

In [ ]:
window_sizes = [1, 2, 3, 5, 10, 15, 20]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

# A. True
cs = axes[0].contourf(x, t_test, y_test_kdv, levels=101, vmin=0, vmax=0.37)
plt.colorbar(cs, ax=axes[0], label='$u(x,t)$')
axes[0].set_xlabel('$x$', fontsize=11)
axes[0].set_ylabel('$t$', fontsize=11)
axes[0].set_title('A. True $u(x,t)$', fontsize=11, fontweight='bold')
axes[0].tick_params(labelsize=9)


labels = ['B', 'C', 'D', 'E', 'F', 'G', 'H']
for idx, w in enumerate(window_sizes):
    xw_list = []
    yw_list = []
    for i in range(len(t) - w):
        xw_list.append(sol[i:i+w, :].flatten())
        yw_list.append(sol[i+w, :])
    xw_list = np.array(xw_list)
    yw_list = np.array(yw_list) 

    xw_train, xw_test, yw_train, yw_test = train_test_split(xw_list, yw_list, test_size=0.3, shuffle=False)

    model = Ridge(alpha=1.0)
    model.fit(xw_train, yw_train)
    yw_pred = model.predict(xw_test)
    r2_w = r2_score(yw_test, yw_pred)
    rmse_w = np.sqrt(mean_squared_error(yw_test, yw_pred))

    split = len(xw_train)
    tw_test = t[-len(yw_pred):]

    ax = axes[idx + 1]
    cs = ax.contourf(x, tw_test, yw_pred, levels=101, vmin=0, vmax=0.37)
    plt.colorbar(cs, ax=ax, label='$u(x,t)$')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$t$')
    ax.set_title(f'{labels[idx]}. window = {w}\n$R^2={r2_w:.3f}$  $RMSE={rmse_w:.3f}$', fontweight='bold')
plt.tight_layout()

## Reference

[1] Lorenz, E. N. (1963). Deterministic nonperiodic flow. Journal of the Atmospheric Sciences, 20(2), 130-141.

[2] Wikipedia contributors. (2026, March 2). Lorenz system. In Wikipedia, The Free Encyclopedia. 

[3] Scikit-learn: Machine Learning in Python, Pedregosa et al., JMLR 12, pp. 2825-2830, 2011.

[4] Wikipedia contributors. (2026, March 15). Korteweg–De Vries equation. In Wikipedia, The Free Encyclopedia. 

[5] Korteweg, D. J., & De Vries, G. (1895). XLI. On the change of form of long waves advancing in a rectangular canal, and on a new type of long stationary waves. The London, Edinburgh, and Dublin Philosophical Magazine and Journal of Science, 39(240), 422-443.

## Statment on AI Usage

The research ideas and methodology of this assignment are original student work. Claude AI was primarily used to enhance code quality, optimize visualization design and improve presentation result. All code was reviewed, validated and tested by the student to ensure correctness. 